In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://devansh@localhost:5432/shipsense_db")

dashboard_query = """
SELECT 
    o.order_id, 
    o.order_purchase_timestamp::timestamp AS purchase_date,
    c.customer_state, 
    c.customer_unique_id,
    pay.total_payment_value,
    o.order_delivered_customer_date::timestamp AS delivered_date,
    o.order_estimated_delivery_date::timestamp AS estimated_date
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
LEFT JOIN (
    SELECT order_id, SUM(payment_value) AS total_payment_value
    FROM order_payments GROUP BY order_id
) pay ON o.order_id = pay.order_id
WHERE o.order_status = 'delivered'
"""
dash_df = pd.read_sql(dashboard_query, engine)
dash_df['days_late'] = (dash_df['delivered_date'] - dash_df['estimated_date']).dt.days
dash_df['is_late'] = (dash_df['days_late'] > 0).astype(int)
dash_df = dash_df.dropna(subset=['total_payment_value'])

dash_df.to_csv('../data_processed/dashboard_data.csv', index=False)
print(f"Rows: {len(dash_df)}")
print(dash_df.head())

Rows: 96477
                           order_id       purchase_date customer_state  \
0  e481f51cbdc54678b7cc49136f2d6af7 2017-10-02 10:56:33             SP   
1  ad21c59c0840e6cb83a9ceb5573f8159 2018-02-13 21:18:39             SP   
2  5ff96c15d0b717ac6ad1f3d77225a350 2018-07-25 17:44:10             SP   
3  432aaf21d85167c2c86ec9448c4e42cc 2018-03-01 14:14:28             SP   
4  dcb36b511fcac050b97cd5c05de84dc3 2018-06-07 19:03:12             GO   

                 customer_unique_id  total_payment_value      delivered_date  \
0  7c396fd4830fd04220f754e42b4e5bff                38.71 2017-10-10 21:25:13   
1  72632f0f9dd73dfee390c9b22eb56dd6                28.62 2018-02-16 18:17:02   
2  e2dfa3127fedbbca9707b36304996dab                32.70 2018-07-30 15:52:25   
3  04cf8185c71090d28baa4407b2e6d600                54.36 2018-03-12 23:36:26   
4  ccafc1c3f270410521c3c6f3b249870f               146.45 2018-06-21 15:34:32   

  estimated_date  days_late  is_late  
0     2017-10-18       